# Prime-power partitions lab

Start with **Recover N** for the current experiment. The later sections run independently after Setup. Edit the inputs in each cell and run it with Shift+Enter. From the project root launch `pixi run jupyter lab labs/prime-power.ipynb`.

## Setup

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import csv, shutil, subprocess
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'scripts/inspect_reciprocity.sage').exists())
S = ROOT/'scripts'
OUT = Path(mkdtemp(prefix='prime-power-lab-'))
load(str(S/'inspect_reciprocity.sage'))
load(str(S/'q_comparison_report.sage'))

def run(*args, cwd=ROOT, stdin=None):
    result = subprocess.run(list(map(str,args)), cwd=cwd, input=stdin, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result.stdout + result.stderr

def records(path):
    with open(path) as stream:
        return list(csv.DictReader(stream))

print("Generated reports and scratch files:", OUT)

## Recover N at 2

The sample 54001 is an odd-ρ control. Set `formal=False` to skip those controls. The full experiment first counts representatives in the broad interval, then uses the adjacent powers of two to narrow the real interval. A singleton from the broad interval is provisional. The two real-candidate flags check that same integer. A missing real candidate means the real interval was already empty.

In [ ]:
q = 54001
source = n_remainder_summary([q], formal=True, silent=True)[0]
case = reverse_n_case(q, source['M'], source['rho'], source['lower'])
case

Inspect each successive residue for N. Set combine to False to see the lift at 2 alone. Each trace row gives the precision of the equation, the exponent residue and the count left in the interval.

In [ ]:
combine = False
trace = reverse_n_lift(q, case['M'], case['e'], case['lower'], case['upper'],
                       remainder=case['N_mod_q_minus_one'] if combine else None, trace=True)
table([['bits', 'N residue', 'period at 2', 'combined period', 'first', 'count']] +
      [list(step.values()) for step in trace['steps']], header_row=True)

Rerun a small prime range below. Use 10^7 to reproduce the full branch-control run. The other saved run takes its inputs from q-n-remainders.csv. The real example q=3, M=2, e=1 recovers N=1 outside the remaining branch.

In [ ]:
limit = 10^5
rows = reverse_n_experiment(reverse_n_branch_primes(limit), formal=True, csv_path=OUT/'reverse.csv')
saved_qs = [ZZ(r['q']) for r in records(ROOT/'data/q-n-remainders.csv')]
saved = reverse_n_experiment(saved_qs, formal=True)
print(len(rows), 'branch controls;', len(saved), 'saved inputs')
reverse_n_case(3, 2, 1, 0)

### Results through 10⁷
Follow the real candidate through the gap restrictions, Nu ≡ −1 modulo 2^c, and the lift at the tested precision.

In [ ]:
def stage_counts(filename):
    controls = records(ROOT/'data'/filename)
    real = [r for r in controls if r['N_real']]
    gaps = [r for r in real if r['real_admissible_gap']=='True']
    linear = [r for r in gaps if r['real_passes_linear']=='True']
    return [len(controls),len(real),len(gaps),len(linear),
            sum(1 for r in linear if r['real_passes_2']=='True')]

labels = ['inputs','real window','admissible gap','linear congruence','lift at tested precision']
table([['stage','formal h, odd ρ','imposed e=2, trial h']] +
      list(zip(labels,stage_counts('reverse-n-branch-controls.csv'),
                      stage_counts('reverse-n-even-controls.csv'))), header_row=True)

For the even-exponent comparison, run the cell below. The full saved run used limit=10^6 and max_h=32. It imposes e=2 and varies h independently of the multiplier reconstruction, retaining the mod 5 case split.

In [ ]:
trial = reverse_n_even_controls(limit=10^5,max_h=8)
print(len(trial), 'trial (q,h) pairs')

## Conditions, multiplier and reciprocity

Change qs to compare different primes. The CSV reports use one row per prime for conditions and one per tested level for reciprocity.

In [ ]:
qs = [433, 1093, 3511]
flags = q_conditions(qs, silent=True, csv_path=OUT/'conditions.csv')
print(dict(zip(Q_CONDITIONS, flags[0])))
print(gap_candidate(1093))
tests = reciprocity_summary(qs, silent=True, csv_path=OUT/'reciprocity.csv')
report = OUT/'comparison.md'
q_comparison_report(OUT/'conditions.csv', OUT/'reciprocity.csv', report, sort_by='q')
print(report)

In [ ]:
list(iter_q_matches(prime_range(97, 1000), required={'mod_240': True}))[:3]

In [ ]:
sample_q_types(OUT/'conditions.csv', top=2, per_type=1)

In [ ]:
inspect_reciprocity(433, 2, factor_ideals=False)

## Native prime scans, prefix search and factorization

The branch scan counts filter survivors. Candidate and wieferich modes emit primes; the default scan includes exploratory rows. The prefix scan fixes n₂ and the inclusive j band, where q has j+1 bits.

In [ ]:
print(run('make','scan_q_native'))
native = ROOT/'build/bin/scan_q_native'
print(run(native,100000,4,'branch'))
print(run(native,10000,2,'wieferich'))
print(run(native,10000,2,'candidate'))

In [ ]:
result = subprocess.run([str(native),'prefix','7','10','100','--compare','--threads','4'],
                        capture_output=True, text=True, check=True)
print(result.stderr)
print(run(native,'factor-candidates',stdin=result.stdout))
print(run(native,'factor',60))

## Forward lift at 2

Read the saved experiment or rebuild the C++ scanner and repeat it. Each row represents one (c,e,N) triple across several M. Set rerun to True for the full grid.

In [ ]:
rerun = False
path = ROOT/'data/2adic-lifts.csv'
if rerun:
    print(run('make','explore_2adic_lifts'))
    binary = ROOT/'build/bin/explore_2adic_lifts'
    path = OUT/'forward.csv'
    with path.open('w') as stream:
        subprocess.run([str(binary)],stdout=stream,check=True)
forward = records(path)
print(len(forward), 'triples;', sum(int(r['windows']) for r in forward), 'windows')
min(forward, key=lambda r:int(r['min_excess_bits']))

## Common gaps and the q | M branch (Lua)

Each check is self-contained. Select any of the three filenames to see its deductions.

In [ ]:
checks = sorted((S/'exp-gcd').glob('*.lua'))
print([p.name for p in checks])
print(run('lua', checks[0]))

## Mersenne factors and order obstructions (GAP)

GAP reads its companion files relative to the working directory. The cell copies the library, factor cache and checks into scratch space so generated tables stay together. Change qMax, nMax and factorE in PPConfig for another table.

In [ ]:
gapdir = OUT/'gap'
gapdir.mkdir(exist_ok=True)
for path in (S/'mers-tab').glob('*.g'):
    shutil.copy(path,gapdir)
def gap_code(instructions):
    return run('gap','-A','-q','-b','--quitonbreak',cwd=gapdir,stdin=instructions+'\nQUIT;\n')
print(gap_code('Read("mersenne_tables_test.g");'))
print(gap_code('PPConfig := rec(qMax:=1000,nMax:=15,factorE:=32,output:="mersenne.md");; Read("mersenne_tables.g");'))

In [ ]:
# Full order-obstruction table using the supplied factor cache.
print(gap_code('Read("check_order_obstruction.g");'))

## Cyclotomic reciprocity (PARI/GP)

Evaluate a single case, then reproduce the bundled checks and Markdown table.

In [ ]:
gpdir = OUT/'gp'
gpdir.mkdir(exist_ok=True)
for path in S.glob('*.gp'):
    shutil.copy(path,gpdir)
print(run('gp','-fq','-s','64M',cwd=gpdir,
          stdin='read("cyclotomic_reciprocity.gp"); print(reciprocity_case(1093,364,13)); quit\n'))
print(run('gp','-fq','-s','64M','check_cyclotomic_reciprocity.gp',cwd=gpdir))